#### Project 3: Autonomous Media Inclusivity Research Agent
##### Day 2: RAG Implementation - Pinecone Vector Database

**Goal:** Build a knowledge base of media inclusivity documents that our agent can search through.

**What is RAG?**
RAG = Retrieval Augmented Generation.
Instead of relying purely on the LLM's training data, we store our own documents in a vector database (Pinecone) and retrieve relevant ones before generating answers.


In [1]:
# CELL 1: Import everything we need for Day 2
# Run this first before anything else

import os
import json
import time
from dotenv import load_dotenv

# Ensure working directory is project root (works whether run from root or notebooks/)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Pinecone v5 imports
from pinecone import Pinecone, ServerlessSpec

# OpenAI for creating embeddings (turning text into vectors)
from openai import OpenAI

# LangChain components for text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load our API keys from .env
load_dotenv()

# Initialize our clients
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Load our project config from Day 1
with open("data/project_config.json", "r") as f:
    config = json.load(f)

print("All imports successful!")
print(f"Project: {config['project_name']}")
print(f"Pinecone index name: {config['pinecone_index_name']}")
print(f"Embedding model: {config['embedding_model']}")

All imports successful!
Project: Autonomous Media Inclusivity Research Agent
Pinecone index name: media-inclusivity-index
Embedding model: text-embedding-ada-002


In [2]:
# CELL 2: Create or connect to the Pinecone index
# media-inclusivity-index / 1536 dims (text-embedding-ada-002) / cosine

INDEX_NAME = config["pinecone_index_name"]   # "media-inclusivity-index"
EMBEDDING_DIM = 1536

existing_names = pc.list_indexes().names()

if INDEX_NAME not in existing_names:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Created index: {INDEX_NAME}")
    time.sleep(10)
else:
    print(f"Index already exists: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(f"Connected to: {INDEX_NAME}")
print(f"Total vectors currently stored: {stats['total_vector_count']}")

Created index: media-inclusivity-index


Connected to: media-inclusivity-index
Total vectors currently stored: 0


In [3]:
# CELL 3: Define the knowledge base documents
# Curated documents covering all 4 research angles

KNOWLEDGE_BASE = [
    # BYLINES AND STORY SELECTION
    {
        "id": "bylines_001",
        "angle": "bylines_and_story_selection",
        "text": (
            "A study of major US newspapers found that women wrote only 37% of front-page stories "
            "despite comprising nearly half the journalism workforce. The disparity is even larger "
            "in sports and politics coverage where male bylines dominate. News organizations that "
            "implement gender-balanced assignment policies see measurable improvements in which "
            "stories get covered and whose perspectives are centered in editorial decisions."
        )
    },
    {
        "id": "bylines_002",
        "angle": "bylines_and_story_selection",
        "text": (
            "The Global Media Monitoring Project found that women are underrepresented as sources, "
            "reporters, and subjects in news media worldwide. Female journalists are more likely to "
            "be assigned to lifestyle and health beats rather than politics, business, and technology. "
            "Organizations that track byline diversity and set transparency benchmarks show stronger "
            "inclusivity outcomes over time through systematic story assignment equity monitoring."
        )
    },
    {
        "id": "bylines_003",
        "angle": "bylines_and_story_selection",
        "text": (
            "Journalists of color account for roughly 17% of newsroom staff at major US outlets "
            "despite representing 40% of the general population. Story selection reflects these gaps: "
            "issues affecting communities of color receive fewer resources, shorter stories, and lower "
            "editorial prominence. Editors who mandate inclusive coverage calendars and monitor story "
            "assignment equity by race and ethnicity report improved representation outcomes."
        )
    },
    {
        "id": "bylines_004",
        "angle": "bylines_and_story_selection",
        "text": (
            "LGBTQ+ journalists in many newsrooms report being steered away from LGBTQ+ stories to "
            "avoid perceived conflicts of interest while cisgender straight journalists face no "
            "equivalent restrictions. This practice limits authentic representation and community "
            "expertise in story selection. Best practices include allowing LGBTQ+ journalists to lead "
            "coverage of their communities while applying the same conflict-of-interest standards used "
            "for all reporters and tracking byline diversity by identity category."
        )
    },
    # PORTRAYAL IN CONTENT
    {
        "id": "portrayal_001",
        "angle": "portrayal_in_content",
        "text": (
            "Research on media coverage of Black Americans shows they are disproportionately featured "
            "as perpetrators of crime rather than as community leaders, experts, or ordinary citizens. "
            "This pattern reinforces racial stereotypes and shapes public perception negatively. "
            "Newsrooms committed to equitable portrayal actively audit crime coverage to ensure all "
            "communities are covered with similar framing focusing on systemic causes rather than "
            "only individual actors from marginalized groups."
        )
    },
    {
        "id": "portrayal_002",
        "angle": "portrayal_in_content",
        "text": (
            "Transgender people in media are frequently reduced to their transition narratives or "
            "portrayed as victims or social controversies rather than as full human beings with complex "
            "lives. The GLAAD Media Reference Guide recommends covering transgender people with the "
            "same dignity as any subject: focusing on their actual expertise, accomplishments, and "
            "perspectives rather than treating their identity as a topic of debate or spectacle."
        )
    },
    {
        "id": "portrayal_003",
        "angle": "portrayal_in_content",
        "text": (
            "Media coverage of disability often employs inspiration porn framing disabled people as "
            "heroic merely for living their daily lives or conversely portraying disability as tragedy. "
            "Disability justice advocates argue for coverage that centers disabled people as experts on "
            "their own lives and systemic barriers rather than as objects of pity or admiration. "
            "Organizations like the National Center on Disability and Journalism publish guidelines for "
            "equitable content portrayal."
        )
    },
    {
        "id": "portrayal_004",
        "angle": "portrayal_in_content",
        "text": (
            "Coverage of women in politics consistently focuses disproportionately on appearance, "
            "emotional state, and personal life compared to coverage of male politicians with equivalent "
            "positions. Studies show this framing reduces perceived competence and authority in the "
            "eyes of audiences. Media organizations that adopt gendered coverage audits and track "
            "portrayal patterns produce more equitable political reporting over time."
        )
    },
    {
        "id": "portrayal_005",
        "angle": "portrayal_in_content",
        "text": (
            "Victim portrayal in crime and disaster coverage frequently dehumanizes marginalized groups "
            "through passive framing, use of mugshots, and focus on vulnerability rather than agency. "
            "Trauma-informed journalism practices recommend centering the voices of affected communities, "
            "obtaining meaningful consent for participation, and following up beyond the initial crisis "
            "to show full human context including recovery, expertise, and community leadership."
        )
    },
    # SOURCING DIVERSITY
    {
        "id": "sourcing_001",
        "angle": "sourcing_diversity",
        "text": (
            "A Nieman Lab analysis of major US newspaper archives found that men are quoted roughly "
            "three times more often than women as expert sources in news coverage. The gap is widest "
            "in stories about economics, science, and national security. Newsrooms that maintain "
            "diverse source databases and track sourcing by demographic category significantly reduce "
            "this disparity over time through systematic sourcing diversity monitoring."
        )
    },
    {
        "id": "sourcing_002",
        "angle": "sourcing_diversity",
        "text": (
            "Inclusive sourcing practices require journalists to seek out experts from underrepresented "
            "communities rather than defaulting to the most easily accessible voices in their existing "
            "networks. The Diverse Sources Database provides searchable directories of women and "
            "minority experts in every major field. Outlets that require reporters to consult diverse "
            "source databases show measurable improvement in sourcing diversity metrics."
        )
    },
    {
        "id": "sourcing_003",
        "angle": "sourcing_diversity",
        "text": (
            "Community members from marginalized groups are frequently quoted only about their personal "
            "experiences with discrimination rather than as technical or policy experts in their fields. "
            "This limits their authority in coverage and reinforces the perception that only "
            "majority-group members hold relevant expertise. Best practices include distinguishing "
            "between community testimony and professional expertise and seeking both from diverse "
            "sources across beats."
        )
    },
    {
        "id": "sourcing_004",
        "angle": "sourcing_diversity",
        "text": (
            "LGBTQ+ experts are rarely sourced on mainstream topics outside of LGBTQ+-specific stories "
            "despite their relevant expertise in law, medicine, policy, and other fields. Anti-LGBTQ+ "
            "organizations are frequently quoted as balance sources in coverage of civil rights issues "
            "creating false equivalence. Ethical sourcing guidelines recommend sourcing actual "
            "subject-matter experts and community representatives rather than political opponents as "
            "balance sources."
        )
    },
    # LANGUAGE AND FRAMING
    {
        "id": "language_001",
        "angle": "language_and_framing",
        "text": (
            "Inclusive language guidelines for journalism recommend using people-first or identity-first "
            "language based on individual preference for disability coverage, avoiding outdated or "
            "clinical terms, and never using the word suffering to describe disability or chronic "
            "illness unless the individual describes their own experience that way. Style guides from "
            "disability organizations provide updated terminology that reflects community preferences "
            "and avoids dehumanizing framing."
        )
    },
    {
        "id": "language_002",
        "angle": "language_and_framing",
        "text": (
            "The AP Stylebook and the GLAAD Media Reference Guide both recommend using a person's "
            "stated name and pronouns in all coverage including when referring to events before their "
            "transition. Using deadnames or incorrect pronouns is considered a serious ethical "
            "violation in contemporary journalism standards. Outlets that have adopted explicit "
            "trans-inclusive language policies report stronger trust from LGBTQ+ audiences and fewer "
            "community complaints about harmful framing."
        )
    },
    {
        "id": "language_003",
        "angle": "language_and_framing",
        "text": (
            "Racial framing in crime coverage frequently uses language that assigns collective blame "
            "to communities of color for individual actions. Terms that imply uniquely pathological "
            "behavior within communities of color while equivalent patterns in white communities go "
            "unnamed are a form of biased framing. The Society of Professional Journalists ethics "
            "guidelines recommend reporting on race only when directly relevant and avoiding language "
            "that reinforces racial stereotypes in news coverage."
        )
    },
    {
        "id": "language_004",
        "angle": "language_and_framing",
        "text": (
            "Gendered language in newsrooms shapes coverage in ways that are often invisible to "
            "practitioners. Terms like female doctor or woman CEO imply these are exceptions rather "
            "than norms. Many outlets have adopted style guides that eliminate gendered modifiers "
            "unless gender is directly relevant to the story. These inclusive language policies extend "
            "to descriptions of physical appearance, emotional reactions, and personal background "
            "applied differently by gender."
        )
    },
    {
        "id": "language_005",
        "angle": "language_and_framing",
        "text": (
            "Framing effects in immigration coverage significantly shape public opinion about immigrant "
            "communities. Research shows that metaphors like waves, floods, and invasions activate fear "
            "responses and dehumanize migrants by using disaster terminology. Outlets committed to "
            "accurate immigration coverage have adopted language policies that use human terms and "
            "provide demographic and legal context rather than relying on crisis framing that "
            "dehumanizes people seeking safety or opportunity."
        )
    }
]

print(f"Knowledge base defined: {len(KNOWLEDGE_BASE)} documents")
print("\nDocuments per research angle:")
angle_counts = {}
for doc in KNOWLEDGE_BASE:
    angle = doc["angle"]
    angle_counts[angle] = angle_counts.get(angle, 0) + 1
for angle, count in sorted(angle_counts.items()):
    print(f"  {angle}: {count} documents")

Knowledge base defined: 18 documents

Documents per research angle:
  bylines_and_story_selection: 4 documents
  language_and_framing: 5 documents
  portrayal_in_content: 5 documents
  sourcing_diversity: 4 documents


In [4]:
# CELL 4: Split documents into chunks
# chunk_size=500, chunk_overlap=50 as specified in project config

splitter = RecursiveCharacterTextSplitter(
    chunk_size=config["chunk_size"],       # 500
    chunk_overlap=config["chunk_overlap"]  # 50
)

chunks = []
for doc in KNOWLEDGE_BASE:
    split_texts = splitter.split_text(doc["text"])
    for i, chunk_text in enumerate(split_texts):
        chunks.append({
            "id": f"{doc['id']}_chunk{i}",
            "angle": doc["angle"],
            "text": chunk_text
        })

print(f"Original documents: {len(KNOWLEDGE_BASE)}")
print(f"Total chunks after splitting: {len(chunks)}")
print("\nSample chunk:")
print(f"  ID: {chunks[0]['id']}")
print(f"  Angle: {chunks[0]['angle']}")
print(f"  Text preview: {chunks[0]['text'][:120]}...")

Original documents: 18
Total chunks after splitting: 19

Sample chunk:
  ID: bylines_001_chunk0
  Angle: bylines_and_story_selection
  Text preview: A study of major US newspapers found that women wrote only 37% of front-page stories despite comprising nearly half the ...


In [5]:
# CELL 5: Create embeddings and upsert to Pinecone
# Only upsert chunks that are not already in the index

existing_stats = index.describe_index_stats()
existing_count = existing_stats["total_vector_count"]

if existing_count >= len(chunks):
    print(f"Index already has {existing_count} vectors -- skipping upsert")
else:
    print(f"Upserting {len(chunks)} chunks to Pinecone...")
    batch_size = 50
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c["text"] for c in batch]

        # Create embeddings via OpenAI
        response = openai_client.embeddings.create(
            model=config["embedding_model"],
            input=texts
        )
        embeddings = [r.embedding for r in response.data]

        # Format for Pinecone upsert
        vectors = [
            {
                "id": batch[j]["id"],
                "values": embeddings[j],
                "metadata": {
                    "angle": batch[j]["angle"],
                    "text": batch[j]["text"]
                }
            }
            for j in range(len(batch))
        ]

        index.upsert(vectors=vectors)
        print(f"  Upserted batch {i // batch_size + 1} ({len(batch)} vectors)")
        time.sleep(1)

    print(f"\nUpsert complete! Total chunks uploaded: {len(chunks)}")

time.sleep(3)
final_stats = index.describe_index_stats()
print(f"Index now contains: {final_stats['total_vector_count']} vectors")

Upserting 19 chunks to Pinecone...


  Upserted batch 1 (19 vectors)



Upsert complete! Total chunks uploaded: 19


Index now contains: 19 vectors


In [6]:
# CELL 6: Verify index is ready with a quick test query

test_query = "how are women represented in newsroom bylines"
test_embedding = openai_client.embeddings.create(
    model=config["embedding_model"],
    input=[test_query]
).data[0].embedding

test_result = index.query(
    vector=test_embedding,
    top_k=3,
    include_metadata=True
)

print("Index ready check:")
print(f"Test query: '{test_query}'")
print(f"Matches returned: {len(test_result['matches'])}")
print("\nTop result:")
if test_result["matches"]:
    top = test_result["matches"][0]
    print(f"  Score: {top['score']:.4f}")
    print(f"  Angle: {top['metadata']['angle']}")
    print(f"  Text: {top['metadata']['text'][:120]}...")

Index ready check:
Test query: 'how are women represented in newsroom bylines'
Matches returned: 3

Top result:
  Score: 0.8754
  Angle: bylines_and_story_selection
  Text: A study of major US newspapers found that women wrote only 37% of front-page stories despite comprising nearly half the ...


In [7]:
# CELL 7: Build the RAG query function
# This is the core retrieval function the agent will use in Day 3

def rag_query(query: str, top_k: int = 3, angle_filter: str = None) -> list:
    """
    Query the Pinecone knowledge base and return relevant documents.

    Args:
        query: The research question to look up
        top_k: Number of results to return
        angle_filter: Optional - filter by research angle

    Returns:
        List of dicts with score, angle, and text
    """
    query_embedding = openai_client.embeddings.create(
        model=config["embedding_model"],
        input=[query]
    ).data[0].embedding

    query_kwargs = {
        "vector": query_embedding,
        "top_k": top_k,
        "include_metadata": True
    }

    if angle_filter:
        query_kwargs["filter"] = {"angle": {"$eq": angle_filter}}

    results = index.query(**query_kwargs)

    return [
        {
            "score": match["score"],
            "angle": match["metadata"]["angle"],
            "text": match["metadata"]["text"]
        }
        for match in results["matches"]
    ]


print("rag_query() function defined.")
print("\nTest: querying for sourcing diversity")
results = rag_query("who gets quoted as expert sources in journalism", top_k=2)
for r in results:
    print(f"  [{r['score']:.4f}] ({r['angle']}) {r['text'][:100]}...")

rag_query() function defined.

Test: querying for sourcing diversity


  [0.8682] (sourcing_diversity) A Nieman Lab analysis of major US newspaper archives found that men are quoted roughly three times m...
  [0.8270] (sourcing_diversity) LGBTQ+ experts are rarely sourced on mainstream topics outside of LGBTQ+-specific stories despite th...


In [8]:
# CELL 8: Count vectors stored in Pinecone

stats = index.describe_index_stats()
total_vectors = stats["total_vector_count"]

print("Pinecone Index Stats")
print("-" * 40)
print(f"Index name:     {INDEX_NAME}")
print(f"Dimensions:     {EMBEDDING_DIM}")
print(f"Total vectors:  {total_vectors}")
print(f"Metric:         cosine")

Pinecone Index Stats
----------------------------------------
Index name:     media-inclusivity-index
Dimensions:     1536
Total vectors:  19
Metric:         cosine


In [9]:
# CELL 9: Test RAG retrieval across all 4 angles
# One representative query per angle

angle_queries = {
    "bylines_and_story_selection": "gender and racial diversity in journalist bylines and story assignments",
    "portrayal_in_content": "how marginalized groups are portrayed as victims or experts in news stories",
    "sourcing_diversity": "diversity of expert sources quoted in journalism and news coverage",
    "language_and_framing": "inclusive language guidelines and framing in media reporting"
}

print("RAG retrieval test across all 4 angles")
print("=" * 50)

for angle, query in angle_queries.items():
    results = rag_query(query, top_k=2)
    print(f"\nAngle: {angle}")
    print(f"Query: {query}")
    for i, r in enumerate(results):
        print(f"  Result {i+1}: score={r['score']:.4f} | angle={r['angle']}")

RAG retrieval test across all 4 angles



Angle: bylines_and_story_selection
Query: gender and racial diversity in journalist bylines and story assignments
  Result 1: score=0.8981 | angle=bylines_and_story_selection
  Result 2: score=0.8932 | angle=bylines_and_story_selection



Angle: portrayal_in_content
Query: how marginalized groups are portrayed as victims or experts in news stories
  Result 1: score=0.8888 | angle=portrayal_in_content
  Result 2: score=0.8727 | angle=sourcing_diversity



Angle: sourcing_diversity
Query: diversity of expert sources quoted in journalism and news coverage
  Result 1: score=0.9028 | angle=sourcing_diversity
  Result 2: score=0.8934 | angle=sourcing_diversity



Angle: language_and_framing
Query: inclusive language guidelines and framing in media reporting
  Result 1: score=0.8503 | angle=language_and_framing
  Result 2: score=0.8461 | angle=language_and_framing


In [10]:
# CELL 10: Verify RAG pipeline is working end to end with LLM synthesis

sample_query = "How do major news outlets handle representation of women in their bylines and story selection?"
retrieved_docs = rag_query(sample_query, top_k=3)
context = "\n\n".join([f"[{r['angle']}]: {r['text']}" for r in retrieved_docs])

response = openai_client.chat.completions.create(
    model=config["llm_model"],
    messages=[
        {
            "role": "system",
            "content": (
                "You are a media inclusivity researcher. Answer questions about media "
                "representation using the provided context. Be specific and evidence-based."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {sample_query}"
        }
    ],
    max_tokens=300
)

print("End-to-end RAG test")
print("=" * 50)
print(f"Query: {sample_query}")
print(f"\nRetrieved {len(retrieved_docs)} documents:")
for r in retrieved_docs:
    print(f"  [{r['score']:.4f}] {r['angle']}")
print(f"\nLLM Response:")
print(response.choices[0].message.content)

End-to-end RAG test
Query: How do major news outlets handle representation of women in their bylines and story selection?

Retrieved 3 documents:
  [0.8983] bylines_and_story_selection
  [0.8893] bylines_and_story_selection
  [0.8724] bylines_and_story_selection

LLM Response:
Major news outlets exhibit significant disparities in the representation of women in their bylines and story selection. According to a study of major US newspapers, women authored only 37% of front-page stories, which is striking given that they make up nearly half of the journalism workforce. This inequity becomes even more pronounced in specific sections such as sports and politics, where male bylines overwhelmingly dominate.

The findings indicate a systematic underrepresentation of women journalists in key areas that shape public discourse, with women often relegated to lifestyle and health coverage instead of more prominent beats like politics, business, and technology. The Global Media Monitoring Project co

In [11]:
# CELL 11: RAG quality test - all 4 angles must score above 0.7

QUALITY_THRESHOLD = 0.7

quality_queries = {
    "bylines_and_story_selection": "gender and racial diversity in journalist bylines and story assignments",
    "portrayal_in_content": "how marginalized groups are portrayed as victims or experts in news coverage",
    "sourcing_diversity": "diversity of expert sources quoted in journalism and news reporting",
    "language_and_framing": "inclusive language guidelines and framing bias in media reporting"
}

print("RAG Quality Test")
print("=" * 50)
print(f"Threshold: {QUALITY_THRESHOLD}")
print()

all_pass = True
scores = {}

for angle, query in quality_queries.items():
    results = rag_query(query, top_k=3)
    if not results:
        print(f"FAIL  {angle}: no results returned")
        all_pass = False
        scores[angle] = 0.0
        continue

    best_score = max(r["score"] for r in results)
    scores[angle] = best_score
    status = "PASS" if best_score >= QUALITY_THRESHOLD else "FAIL"
    if best_score < QUALITY_THRESHOLD:
        all_pass = False
    print(f"{status}  {angle}")
    print(f"      best score: {best_score:.4f} (threshold: {QUALITY_THRESHOLD})")

print()
print("=" * 50)
if all_pass:
    print("ALL PASS - RAG quality test passed for all 4 angles")
else:
    print("SOME ANGLES FAILED - check scores above")
    failing = [a for a, s in scores.items() if s < QUALITY_THRESHOLD]
    print(f"Failing angles: {failing}")

RAG Quality Test
Threshold: 0.7



PASS  bylines_and_story_selection
      best score: 0.8981 (threshold: 0.7)


PASS  portrayal_in_content
      best score: 0.8920 (threshold: 0.7)


PASS  sourcing_diversity
      best score: 0.9024 (threshold: 0.7)


PASS  language_and_framing
      best score: 0.8487 (threshold: 0.7)

ALL PASS - RAG quality test passed for all 4 angles


In [12]:
# CELL 12: Save the RAG pipeline as a reusable module in src/

rag_pipeline_code = '''"""
RAG pipeline for the Media Inclusivity Research Agent.
Auto-generated by day2_rag.ipynb Cell 12.
"""

import os
import json
from dotenv import load_dotenv
from pinecone import Pinecone
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

with open("data/project_config.json", "r") as f:
    config = json.load(f)

index = pc.Index(config["pinecone_index_name"])


def rag_query(query: str, top_k: int = 3, angle_filter: str = None) -> list:
    """
    Query the Pinecone knowledge base and return relevant documents.

    Args:
        query: The research question to look up
        top_k: Number of results to return
        angle_filter: Optional - filter by research angle
            Options: bylines_and_story_selection, portrayal_in_content,
                     sourcing_diversity, language_and_framing

    Returns:
        List of dicts with keys: score, angle, text
    """
    query_embedding = openai_client.embeddings.create(
        model=config["embedding_model"],
        input=[query]
    ).data[0].embedding

    query_kwargs = {
        "vector": query_embedding,
        "top_k": top_k,
        "include_metadata": True
    }

    if angle_filter:
        query_kwargs["filter"] = {"angle": {"$eq": angle_filter}}

    results = index.query(**query_kwargs)

    return [
        {
            "score": match["score"],
            "angle": match["metadata"]["angle"],
            "text": match["metadata"]["text"]
        }
        for match in results["matches"]
    ]
'''

rag_pipeline_path = "src/rag_pipeline.py"
with open(rag_pipeline_path, "w") as f:
    f.write(rag_pipeline_code)

print(f"Saved: {rag_pipeline_path}")
print("\nContent preview:")
print(rag_pipeline_code[:400] + "...")

Saved: src/rag_pipeline.py

Content preview:
"""
RAG pipeline for the Media Inclusivity Research Agent.
Auto-generated by day2_rag.ipynb Cell 12.
"""

import os
import json
from dotenv import load_dotenv
from pinecone import Pinecone
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

with open("data/project_config.json", "r") as f:
    co...
